{ “cells”: \[ { “cell_type”: “markdown”, “id”: “cb55fff9”, “metadata”:
{}, “source”: \[ “\# 06_symbolic_forecasting.ipynb”, “”, “This notebook
builds a symbolic polynomial forecasting model using SageMath. You can
experiment with:”, “- Degree of the polynomial”, “- Number of training
samples”, “- Prediction horizon”, “- Evaluation accuracy” \] }, {
“cell_type”: “markdown”, “id”: “7fb15133”, “metadata”: {}, “source”: \[
“\## 1. Setup and Data Load” \] }, { “cell_type”: “code”,
“execution_count”: 1, “id”: “811b31e3”, “metadata”: {}, “outputs”: \[\],
“source”: \[ “from sage.all import \*“,”import pandas as pd“,”import
matplotlib.pyplot as plt“,”from datetime import timedelta“,”import numpy
as np“,”“,”\# Load BTC time series“,”df =
pd.read_csv(‘../data/bitcoin_timeseries.csv’)“,”df\[‘timestamp’\] =
pd.to_datetime(df\[‘timestamp’\])“,”df =
df.sort_values(‘timestamp’).set_index(‘timestamp’)” \] }, { “cell_type”:
“markdown”, “id”: “d1b1f731”, “metadata”: {}, “source”: \[ “\## 2.
Define Hyperparameters” \] }, { “cell_type”: “code”, “execution_count”:
2, “id”: “04acbdb9”, “metadata”: {}, “outputs”: \[\], “source”: \[ “\#
Hyperparameters - use native Python integers to avoid type issues”,
“days_to_use = 7 \# trailing days of data”, “resample_freq = ‘1h’ \#
hourly”, “samples_to_use = 100 \# how many points from tail to use”,
“degree = 5 \# degree of the polynomial”, “forecast_hours = 12 \#
prediction horizon” \] }, { “cell_type”: “markdown”, “id”: “b54bd832”,
“metadata”: {}, “source”: \[ “\## 3. Prepare Training Data” \] }, {
“cell_type”: “code”, “execution_count”: 3, “id”: “70f9a07f”, “metadata”:
{}, “outputs”: \[\], “source”: \[ “\# 1. Select recent data”, “end_time
= df.index.max()”, “start_time = end_time -
pd.Timedelta(days=days_to_use) \# Use Python int”, “recent =
df.loc\[(df.index \>= start_time) & (df.index \<= end_time)\]”, “”, “\#
2. Resample and clean”, “df_model =
recent.resample(resample_freq).mean().dropna().reset_index()”, “”, “\#
3. Select last N samples”, “df_model = df_model.tail(samples_to_use) \#
Use Python int”, “”, “\# 4. Normalize time axis (in hours from first
point, centered)”, “timestamps = df_model\[‘timestamp’\]”, “t0 =
timestamps.iloc\[0\] \# Use Python int”, “x_vals = \[(t -
t0).total_seconds() / 3600 for t in timestamps\] \# Use Python int”,
“x_mean = sum(x_vals) / len(x_vals)”, “x_vals = \[x - x_mean for x in
x_vals\]”, “”, “\# 5. Target values”, “y_vals =
df_model\[‘price_usd’\].tolist()” \] }, { “cell_type”: “markdown”, “id”:
“9f015bff”, “metadata”: {}, “source”: \[ “\## 4. Symbolic Polynomial
Fitting” \] }, { “cell_type”: “code”, “execution_count”: 4, “id”:
“7a8a2197”, “metadata”: {}, “outputs”: \[\], “source”: \[ “\# Define
symbolic model”, “x = var(‘x’)”, “params = list(var(\[f’a{i}’ for i in
range(degree + 1)\]))”, “model = sum(params\[i\] \* x\*\*i for i in
range(degree + 1))“,”“,”\# Fit the model“,”points = list(zip(x_vals,
y_vals))“,”fit = find_fit(points, model, parameters=params,
variables=\[x\], solution_dict=True)“,”model_fitted =
model.subs(fit)“,”“,”\# Evaluate on training data“,”f = lambda t:
float(model_fitted.subs(x=t))“,”y_fit = \[f(t) for t in x_vals\]” \] },
{ “cell_type”: “markdown”, “id”: “77e216e6”, “metadata”: {}, “source”:
\[ “\## 5. Forecast Future Values” \] }, { “cell_type”: “code”,
“execution_count”: 5, “id”: “b568e860”, “metadata”: {}, “outputs”: \[\],
“source”: \[ “\# 1. Generate future x values with Python native types”,
“last_x = float(x_vals\[-1\]) \# Ensure Python float”, “future_x =
\[last_x + float(i) for i in range(1, forecast_hours + 1)\]”,
“future_preds = \[f(t) for t in future_x\]”, “”, “\# 2. Generate future
timestamps with Python native types”, “\# Convert to Python list to
avoid SageMath issues”, “timestamps_list =
list(df_model\[‘timestamp’\])”, “last_time = timestamps_list\[-1\] \#
Get the last element using Python indexing”, “future_times =
\[last_time + timedelta(hours=i) for i in range(1, forecast_hours +
1)\]”, “”, “\# 3. Build forecast DataFrame”, “df_future =
pd.DataFrame({”, ” ‘timestamp’: future_times,“,” ‘predicted_price’:
future_preds“,”})” \] }, { “cell_type”: “markdown”, “id”: “0f7de82c”,
“metadata”: {}, “source”: \[ “\## 6. Plot Historical and Forecasted
Values” \] }, { “cell_type”: “code”, “execution_count”: 6, “id”:
“46864964”, “metadata”: {}, “outputs”: \[\], “source”: \[
“plt.figure(figsize=(12, 5))”, “plt.plot(df_model\[‘timestamp’\],
y_vals, label=‘Historical Price’, marker=‘o’)”,
“plt.plot(df_model\[‘timestamp’\], y_fit, label=‘Fitted Curve’,
linestyle=‘–’)”, “plt.plot(df_future\[‘timestamp’\],
df_future\[‘predicted_price’\], label=‘Forecast’, marker=‘x’,
linestyle=‘–’)”, “plt.title(f"BTC Price Forecast using Degree-{degree}
Symbolic Polynomial")”, “plt.xlabel("Time")”, “plt.ylabel("Price
(USD)")”, “plt.grid(True)”, “plt.legend()”, “plt.tight_layout()”,
“plt.savefig(‘../reports/symbolic_forecast_polydeg{}.png’.format(degree))”,
“plt.show()” \] }, { “cell_type”: “markdown”, “id”: “c8214c28”,
“metadata”: {}, “source”: \[ “\## 7. Evaluate Fit Accuracy on Training
Data” \] }, { “cell_type”: “code”, “execution_count”: 7, “id”:
“910c9868”, “metadata”: {}, “outputs”: \[\], “source”: \[ “try:”, ” from
sklearn.metrics import mean_absolute_error, mean_squared_error“,” “,” \#
Convert to Python native lists in case of any SageMath types“,”
y_vals_py = \[float(y) for y in y_vals\]“,” y_fit_py = \[float(y) for y
in y_fit\]“,” “,” mae = mean_absolute_error(y_vals_py, y_fit_py)“,” rmse
= mean_squared_error(y_vals_py, y_fit_py, squared=False)“,” “,”
print(f"📏 MAE (Mean Absolute Error): {mae:.2f} USD")“,” print(f"📏 RMSE
(Root Mean Squared Error): {rmse:.2f} USD")“,” “,”except ImportError:“,”
\# Manual calculation if sklearn is not available“,” y_vals_py =
\[float(y) for y in y_vals\]“,” y_fit_py = \[float(y) for y in
y_fit\]“,” “,” \# Calculate MAE manually“,” mae = sum(abs(y_true -
y_pred) for y_true, y_pred in zip(y_vals_py, y_fit_py)) /
len(y_vals_py)“,” “,” \# Calculate RMSE manually“,” rmse =
(sum((y_true - y_pred)**2 for y_true, y_pred in zip(y_vals_py,
y_fit_py)) / len(y_vals_py))**0.5“,” “,” print(f"📏 MAE (Mean Absolute
Error): {mae:.2f} USD")“,” print(f"📏 RMSE (Root Mean Squared Error):
{rmse:.2f} USD")” \] }, { “cell_type”: “markdown”, “id”: “a8e512b3”,
“metadata”: {}, “source”: \[ “\## 8. Print Symbolic Expression” \] }, {
“cell_type”: “code”, “execution_count”: 8, “id”: “e9c428f0”, “metadata”:
{}, “outputs”: \[\], “source”: \[ “\# Pretty print the symbolic
expression”, “print("\nPolynomial Expression:")”, “print(model_fitted)”,
“”, “\# Print coefficients separately”, “print("\nCoefficients:")”, “for
i in range(degree + 1):”, ” coef = fit.get(var(f’a{i}’), 0)“,” print(f"
a{i} = {coef}")” \] } \], “metadata”: { “kernelspec”: { “display_name”:
“SageMath 10.6”, “language”: “sage”, “name”: “sagemath” },
“language_info”: { “codemirror_mode”: { “name”: “ipython”, “version”: 3
}, “file_extension”: “.py”, “mimetype”: “text/x-python”, “name”:
“python”, “nbconvert_exporter”: “python”, “pygments_lexer”: “ipython3”,
“version”: “3.12.5” } }, “nbformat”: 4, “nbformat_minor”: 5 }